# scqubits example: efficient parameter sweeps (`num_cpus` and BLAS threads)

J. Koch and P. Groszkowski

For further documentation of scqubits see https://scqubits.readthedocs.io/en/latest/.

---

`ParameterSweep` can compute a sweep in parallel across worker processes via its
`num_cpus` argument. How much that helps — and whether it helps at all — depends on
**two interacting knobs**:

1. **`num_cpus`** — how many worker processes share the grid points.
2. **BLAS/LAPACK threads** — every eigensolve (via `numpy`/`scipy`) runs on a
   multithreaded linear-algebra backend (OpenBLAS or MKL) that by default uses *all*
   cores.

Running `num_cpus` workers that each launch a full BLAS thread pool oversubscribes the
cores; and on the small matrices typical of qubit truncations, a heavily threaded BLAS
can be *slower* than a single thread. The fastest setting is a balance — roughly
`num_cpus × BLAS-threads ≈ number of cores` — and it depends on your machine, BLAS
library, and problem size.

**This notebook shows how to measure it on your own system.** The timing cells are left
unexecuted on purpose: run them yourself — your numbers will differ.

## 1. Set the BLAS thread count *before* importing scqubits

The BLAS backend reads its thread count once, at import time. So this must be the
**first** cell you run in a fresh kernel — before `numpy` or `scqubits` is imported. (If
you have already imported scqubits in this kernel, restart it first.)

Capping to a small number — often `1` — is a good starting point for qubit sweeps, and
is also what prevents oversubscription once `num_cpus > 1`.

In [ ]:
import os

# Must run before numpy / scqubits are imported (restart the kernel otherwise).
for _var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):
    os.environ[_var] = "1"

In [ ]:
import time

import numpy as np

import scqubits as scq

## 2. A coupled example system

Two tunable transmons coupled to a common resonator (the same system as
`demo_parametersweep`), swept over `flux` and `ng`. We wrap construction in a factory so
we can rebuild a fresh sweep for each timed run.

In [ ]:
def make_sweep(num_cpus, flux_points=11):
    tmon1 = scq.TunableTransmon(
        EJmax=40.0, EC=0.2, d=0.1, flux=0.0, ng=0.3, ncut=40, truncated_dim=3
    )
    tmon2 = scq.TunableTransmon(
        EJmax=15.0, EC=0.15, d=0.2, flux=0.0, ng=0.0, ncut=30, truncated_dim=3
    )
    resonator = scq.Oscillator(E_osc=4.5, truncated_dim=4)

    hilbertspace = scq.HilbertSpace([tmon1, tmon2, resonator])
    hilbertspace.add_interaction(
        g_strength=0.1, op1=tmon1.n_operator, op2=resonator.creation_operator, add_hc=True
    )
    hilbertspace.add_interaction(
        g_strength=0.2, op1=tmon2.n_operator, op2=resonator.creation_operator, add_hc=True
    )

    flux_vals = np.linspace(0.0, 2.0, flux_points)
    ng_vals = np.linspace(-0.5, 0.5, 3)

    def update_hilbertspace(flux, ng):
        tmon1.flux = flux
        tmon2.flux = 1.2 * flux
        tmon2.ng = ng

    return scq.ParameterSweep(
        hilbertspace=hilbertspace,
        paramvals_by_name={"flux": flux_vals, "ng": ng_vals},
        update_hilbertspace=update_hilbertspace,
        subsys_update_info={"flux": [tmon1, tmon2], "ng": [tmon2]},
        evals_count=20,
        num_cpus=num_cpus,
        autorun=False,
    )

## 3. Time `ParameterSweep.run()` versus `num_cpus`

Wall-clock timing is noisy, so we take the median of a few repeats and discard a warm-up
run (the first parallel run pays a one-time process-startup cost). **Results are
machine-specific** — what matters is the *shape* of the trend on your hardware, not any
single number.

In [ ]:
def time_run(num_cpus, flux_points=11, repeats=3):
    make_sweep(num_cpus, flux_points).run()  # warm-up (discarded)
    times = []
    for _ in range(repeats):
        sweep = make_sweep(num_cpus, flux_points)
        start = time.perf_counter()
        sweep.run()
        times.append(time.perf_counter() - start)
    return float(np.median(times))


cores = os.cpu_count() or 1
for n in [c for c in (1, 2, 4) if c <= cores]:
    print(f"num_cpus={n}:  {time_run(n):.3f} s (median)")

## 4. See the BLAS-threading effect

To feel the impact of the thread cap, **restart the kernel**, change the cap in the first
cell from `"1"` to e.g. `str(os.cpu_count())` (let BLAS use every core), and re-run the
timing. On small-matrix sweeps you will typically find the capped version is faster even
at `num_cpus=1`, because an un-capped BLAS spends time coordinating threads on matrices
too small to benefit.

## 5. Larger sweeps and automated tuning

The benefit of `num_cpus > 1` grows with the number of grid points — more work to
amortize the fixed startup cost. Compare a bigger grid to see parallelism pay off:

```python
print("num_cpus=1:", time_run(1, flux_points=64))
print("num_cpus=4:", time_run(4, flux_points=64))
```

Because the optimum depends on your machine, BLAS library, and problem, the most reliable
approach is to measure. If you work from the scqubits **source tree**, the script
`tools/autotune_multiprocessing.py` automates this: it searches the
`num_cpus × BLAS-threads` frontier and reports the fastest configuration for your system.

## Summary

- Two knobs interact: `num_cpus` (worker processes) and BLAS/LAPACK threads.
- Set BLAS threads **before** importing scqubits; capping low (often `1`) is a good
  default for qubit sweeps and prevents oversubscription when `num_cpus > 1`.
- Keep `num_cpus × BLAS-threads ≈ cores`; raise `num_cpus` only for large grids.
- Measure on your own hardware — timings are machine-specific.